## WaveNet-style CNN

Below is an implementation of a model following the "wave" architecture as seen in Google DeepMind's [WaveNet](https://arxiv.org/pdf/1609.03499) paper. We start with a sequence of 8 tokens, and slowly flatten it to

In [2]:
from datasets import load_dataset

ds = load_dataset("parquet",
                    data_files={'train': 'train.parquet',
                                "validation" : "validation.parquet",
                                'test': 'test.parquet'}
                    )

Generating train split: 0 examples [00:00, ? examples/s]

Generating validation split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

In [3]:
text_split = []

for row in ds["train"]["sentence"]:
    for w in row.split():
        text_split.append(w)

text_split = set(text_split)

In [4]:
s_i = {s:i+1 for i, s in enumerate(text_split)}
s_i["<n>"] = 0

i_s = {i:s for s, i in s_i.items()}

In [5]:
import torch

# building dataset

block_size = 8

def build_dataset(sents):


    X, Y = [], []
    for s in sents:
        #print(s)
        context = [0] * block_size
        s[-1] = "<n>"

        for w in s:

            ix = s_i[w]
            X.append(context)
            Y.append(ix)
            # print("".join(i_s[i] for i in context), "---->", i_s[ix])
            context = context[1:] + [ix]

    X = torch.tensor(X)
    Y = torch.tensor(Y)
    return X, Y

In [6]:
sents_split = [s.split() for s in ds["train"]["sentence"]]

Xtr, ytr = build_dataset(sents_split)

In [7]:
dstr = torch.utils.data.TensorDataset(Xtr, ytr)

trloader = torch.utils.data.DataLoader(
    dataset=dstr,
    batch_size=500,
    shuffle=True,

)

In [33]:
# from torch_xla.core.xla_model import xm

""" class Reshape(torch.nn.Module):
    def __call__(self, x):
        # print(x.shape[0] / 2)
        if x.dim() == 2:
            return x.view((int(x.shape[0] / 2), -1))
        else:
            return torch.reshape(x, (int(x.shape[0]), int(x.shape[1] / 8), -1))

class Squeeze(torch.nn.Module):
    def __call__(self, x):
        return  """

class NN(torch.nn.Module):
    def __init__(self, vocab_size, emb_dim, n_hidden):
        super().__init__()

        self.vocab_size = vocab_size
        """
        self.layers = [
            torch.nn.Embedding(vocab_size, emb_dim),
            torch.nn.Linear(emb_dim, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Reshape(), torch.nn.Linear(n_hidden * 2, n_hidden), torch.nn.LayerNorm(n_hidden), torch.nn.Tanh(),
            Squeeze(), torch.nn.Linear(n_hidden, vocab_size)
        ] """

        self.embedding = torch.nn.Embedding(vocab_size, emb_dim)
        self.rnn = torch.nn.RNN(input_size=emb_dim, hidden_size=n_hidden, num_layers=4, nonlinearity=)
        # self.reshape = Reshape()
        # self.squeeze = Squeeze()
        self.l1 = torch.nn.Linear(n_hidden * 8, vocab_size)

        self.out = 0.0

        self.parameters_ = [p for p in self.embedding.parameters()] + [p for p in self.rnn.parameters()] + [p for p in self.l1.parameters()]

    """ def parameters(self):
        params = []

        for layer in self.layers:
            for p in layer.parameters:
                params.append(p)

        return params """

    def __call__(self, x, hn):
        x_ = x.to(device)

        x_ = self.embedding(x)
        # print(x_.shape)

        x_, hn = self.rnn(x_, hn)
        # print(x_.shape)

        single = True if x_.dim() == 2 else False

        x_ = x_.view((int(x_.shape[0] / 2), -1)) if single else torch.reshape(x_, (int(x_.shape[0]), int(x_.shape[1] / 8), -1))
        # print(x_.shape)

        x_ = torch.squeeze(x_)

        x_ = self.l1(x_)
        # print(x_.shape)

        self.out = x_
        return x_, hn

    def fit(self, max_iter, loader, lr):
        g = torch.Generator().manual_seed(2147483647)
        optimizer = torch.optim.AdamW(self.parameters_, lr=lr)

        lossi = []

        for p in self.parameters_:
            p.retain_grad()

        hn = None

        for step in range(max_iter):
            Xb, yb = next(iter(loader))

            Xb = Xb.to(device)
            yb = yb.to(device)

            if hn is not None:
                hn = hn.detach()

            logits, hn = self.__call__(Xb, hn)

            loss = torch.nn.functional.cross_entropy(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            lossi.append(loss.item())

            print(f"\r{step} / {max_iter}: {loss:.6f}", end="", flush=True)

        return lossi


In [41]:
embedding_dim = 32

n_hidden = 100
vocab_size = len(s_i)

net = NN(vocab_size=vocab_size, emb_dim=embedding_dim, n_hidden=n_hidden)


In [35]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device.type)

cuda


In [43]:
net.to(device);

In [11]:
rnn = torch.nn.RNN(10, 20, 2)
input = torch.randn(5, 3, 10)
h0 = torch.randn(2, 3, 20)

In [12]:
# net(Xtr[1], None)
net(Xtr[10].clone().expand(1, 8).to(device), None)

(tensor([ 0.0149, -0.0022, -0.1154,  ...,  0.0598,  0.0469, -0.0331],
        device='cuda:0', grad_fn=<ViewBackward0>),
 tensor([[[-0.2926,  0.3142, -0.1721,  ...,  0.1188,  0.3214,  0.0960],
          [ 0.1443, -0.2993,  0.2195,  ...,  0.0262,  0.5090, -0.5842],
          [-0.2708,  0.5982,  0.5498,  ...,  0.4469, -0.0378, -0.1477],
          ...,
          [-0.0461,  0.2194, -0.0699,  ...,  0.3140, -0.1413, -0.5436],
          [-0.2509,  0.6209,  0.0806,  ..., -0.0151,  0.5649,  0.4581],
          [-0.5090,  0.3147, -0.3134,  ...,  0.5062, -0.2747, -0.1172]],
 
         [[-0.0871, -0.2505, -0.2212,  ...,  0.3099, -0.0961, -0.4308],
          [-0.0089, -0.0149, -0.1502,  ...,  0.1857,  0.1036, -0.2703],
          [-0.2910,  0.2333, -0.3126,  ..., -0.0780,  0.0010, -0.2688],
          ...,
          [ 0.1372,  0.2087, -0.0814,  ...,  0.0065, -0.3007, -0.0152],
          [-0.3869,  0.1661, -0.1637,  ..., -0.1043, -0.3149, -0.3858],
          [-0.0546,  0.2965, -0.2761,  ..., -0.2442, -

In [46]:
lossi = net.fit(max_iter=10000,  loader=trloader, lr=(2.5 * 1e-3));

9999 / 10000: 3.953875

In [45]:
1e-3 * 2.5

0.0025

In [15]:
Xval, yval = build_dataset(s.split() for s in ds["validation"]["sentence"])
dsval = torch.utils.data.TensorDataset(Xval, yval)

valloader = torch.utils.data.DataLoader(dsval, batch_size=100)

In [22]:
def eval_model(model, loader, hn=None):
    total_loss = 0.0

    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = y.to(device)
            logits, h1 = model(x, hn)
            loss = torch.nn.functional.cross_entropy(logits, y)

            total_loss += loss.item() * x.size(0)

    return total_loss / len(loader.dataset)

In [47]:
print("Training set loss:", eval_model(net, trloader))
print("Validation set loss:", eval_model(net, valloader))

Training set loss: 4.030533091083208
Validation set loss: 5.92926952465977


In [39]:
g2 = torch.Generator(device).manual_seed(2147483647)
hni = None

for _ in range(5):
    out = []
    context = [0] * block_size


    while True:
        context = torch.tensor(context).to(device).expand(1, 8)
        probs, hni = net(context, hn=hni)
        logits = torch.nn.functional.softmax(probs, dim=0)

        ix = torch.multinomial(logits, num_samples=1, generator=g2).item()

        context = list(context)[1:] + [ix]

        if ix == 0:
            break

        out.append(ix)

        # if len(out) > 5:
        #     break
        # print(i_s[ix])


    print("".join(i_s[i] + " " for i in out))


signed than development the company to by N N N corporate <unk> a commission in assets for debt supporting & new multibillion-dollar 
he customers of britain assets 's outlook company security of the law of the transactions financing will be such & price from a year any private with 30-share placed as <unk> on <unk> of developing staff 
the exchange preferred to enter in time matters to pop and unisys that you than with kobe in the industry <unk> makers <unk> <unk> <unk> <unk> <unk> <unk> systems series on 
troublesome at N N N N N N N bankruptcy involvement with phase says the decade to drexel 
shearson three security an attorney its 
